# MCDI502 · Evaluación Sumativa 4 · Taller III
## Integración, limpieza, validación y análisis de ventas con Apache Spark en Databricks
**Grupo 2:** Verónica Durán · Raúl Moya · Daniela Rojas · Manuel Sánchez  
**Fuentes:** `ventas_concepcion.csv`, `ventas_santiago.csv`, `ventas_vina_del_mar.csv`  
**Periodo registrado:** octubre–diciembre de 2024 (según los archivos originales).

**Objetivo.** Integramos las ventas de tres sucursales, corregimos problemas de calidad, verificamos reglas de negocio y obtenemos indicadores mediante Spark SQL. El notebook está diseñado para ejecutarse en un *workspace* de Databricks con una sesión Spark disponible.

**Nota:** Los archivos `ventas_region_norte.csv` y `ventas_region_sur.csv` no forman parte de esta evaluación.

## 0. Preparación del entorno y archivos

**Grupo 2:** Verónica Durán, Raúl Moya, Daniela Rojas y Manuel Sánchez.

El notebook funciona en **VS Code** (pruebas locales) y en **Databricks** (entrega oficial). En VS Code, colocamos los tres archivos CSV en una carpeta `datos` junto al notebook, o en `Sumativa 4 - Taller III/datos`. El código busca ambas ubicaciones y también `Descargas` para facilitar el inicio. **No utiliza contraseñas ni rutas personales fijas.**

En Databricks, cargamos los CSV a un volumen de Unity Catalog o a DBFS y actualizamos el widget `ruta_csv` con la carpeta real. Solo se utilizan los archivos `ventas_concepcion.csv`, `ventas_santiago.csv` y `ventas_vina_del_mar.csv`. Los archivos regionales norte/sur no corresponden a esta evaluación.

> El desarrollo y la ejecución final deben realizarse en Databricks, según las instrucciones de la asignatura.

In [ ]:
from pathlib import Path
from functools import reduce
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

ARCHIVOS_REQUERIDOS = (
    "ventas_vina_del_mar.csv", "ventas_santiago.csv", "ventas_concepcion.csv"
)

# Databricks proporciona dbutils; en VS Code no existe.
try:
    dbutils
    EN_DATABRICKS = True
except NameError:
    EN_DATABRICKS = False

if EN_DATABRICKS:
    try:
        spark
    except NameError:
        spark = SparkSession.builder.getOrCreate()
    dbutils.widgets.text("ruta_csv", "/Volumes/workspace/default/ventas_sumativa4", "Carpeta CSV")
    RUTA_CSV = dbutils.widgets.get("ruta_csv").strip().rstrip("/")
    if not RUTA_CSV:
        raise ValueError("En Databricks indica la carpeta real de los CSV en el widget ruta_csv")
else:
    spark = (SparkSession.builder.master("local[2]")
             .appName("MCDI502_Sumativa4_Grupo2").getOrCreate())

    # Primero se busca en la carpeta actual y sus subcarpetas habituales.
    actual = Path.cwd()
    candidatos = [actual / "datos", actual, actual / "Sumativa 4 - Taller III" / "datos"]
    for padre in list(actual.parents)[:4]:
        candidatos.extend([padre / "datos", padre / "Sumativa 4 - Taller III" / "datos"])
    candidatos.extend([Path.home() / "Downloads", Path.home() / "Descargas"])
    carpeta_encontrada = next(
        (carpeta for carpeta in candidatos
         if all((carpeta / archivo).is_file() for archivo in ARCHIVOS_REQUERIDOS)), None
    )
    if carpeta_encontrada is None:
        raise FileNotFoundError(
            "No se encontraron los tres CSV juntos. Crea la carpeta 'datos' junto al notebook "
            "y copia allí ventas_vina_del_mar.csv, ventas_santiago.csv y ventas_concepcion.csv. "
            f"Carpeta actual: {actual}"
        )
    RUTA_CSV = carpeta_encontrada.resolve().as_posix()

print("Entorno:", "Databricks" if EN_DATABRICKS else "VS Code local")
print("Versión Spark:", spark.version)
print("Ruta CSV:", RUTA_CSV)


## 1. Carga e integración de datos (ID3.1)
Cargamos cada sucursal por separado, verificamos su estructura y realizamos la unión por **nombre de columna**, evitando depender del orden de los campos. Conservamos inicialmente los precios y fechas como texto para corregirlos de manera controlada.

In [ ]:
archivos = {
    "Viña del Mar": "ventas_vina_del_mar.csv",
    "Santiago": "ventas_santiago.csv",
    "Concepción": "ventas_concepcion.csv",
}

def leer_csv(nombre):
    ruta = f"{RUTA_CSV}/{nombre}"
    try:
        df = (spark.read.option("header", True)
              .option("inferSchema", False).option("encoding", "UTF-8").csv(ruta))
        df.limit(1).collect()  # Forzar lectura y detectar errores de ruta.
        columnas = {"id_transaccion", "fecha", "producto_nombre", "categoria",
                    "cantidad", "precio_unitario", "sucursal"}
        faltantes = columnas - set(df.columns)
        if faltantes:
            raise ValueError(f"Faltan columnas en {nombre}: {sorted(faltantes)}")
        return df
    except Exception as e:
        raise RuntimeError(
            f"No se pudo leer {ruta}. Comprueba que los tres CSV estén juntos en la carpeta configurada."
        ) from e

df_vina = leer_csv(archivos["Viña del Mar"])
df_santiago = leer_csv(archivos["Santiago"])
df_concepcion = leer_csv(archivos["Concepción"])

for nombre, df in [("Viña del Mar", df_vina), ("Santiago", df_santiago),
                   ("Concepción", df_concepcion)]:
    print(f"\n=== {nombre} ===")
    df.printSchema()
    df.show(5, truncate=False)


In [ ]:
conteos_origen = {"Viña del Mar": df_vina.count(), "Santiago": df_santiago.count(), "Concepción": df_concepcion.count()}
df_ventas_unificado = df_vina.unionByName(df_santiago).unionByName(df_concepcion)
filas_unificadas = df_ventas_unificado.count()
print("Registros por sucursal:", conteos_origen)
print("Suma original:", sum(conteos_origen.values()))
print("Registros unificados:", filas_unificadas)
assert filas_unificadas == sum(conteos_origen.values()), "La unión perdió o duplicó filas"
print("Integridad de unión: OK")

## 2. Diagnóstico y limpieza (ID4.2)
**Decisiones de limpieza:**
- **Nulos en `categoria`:** imputamos `Sin categoría` (`fillna` equivalente). Eliminar esas 46 filas haría perder ventas reales; la etiqueta deja explícito que la clasificación no está confirmada.
- **Duplicados:** eliminamos solo los registros **idénticos en todas las columnas** (`dropDuplicates`), antes de transformar, para no fusionar ventas distintas que se parezcan tras normalizar el texto.
- **Nombres de producto:** pasamos a minúsculas, unificamos espacios y **quitamos tildes** (`láser` → `laser`). Además unificamos **sinónimos** del mismo artículo (`laptop gaming` → `laptop gamer`, `ram 16gb` → `memoria ram 16gb`, `disco ssd 1tb` → `ssd 1tb`, `monitor 27"` → `monitor 27 pulgadas`). Solo con minúsculas quedaban 17 nombres para 10 productos reales.
- **Fechas:** los archivos combinan cuatro formatos (`yyyy-MM-dd`, `yyyy/MM/dd`, `dd/MM/yyyy`, `dd-MM-yyyy`); identificamos el patrón de cada valor y lo convertimos a `DATE`.
- **Precios:** retiramos `$` y espacios antes de convertir a `double`.

In [ ]:
def resumen_calidad(df, etiqueta):
    total = df.count()
    nulos = df.select(*[
        F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), 1).otherwise(0)).alias(c)
        for c in df.columns
    ]).first().asDict()
    duplicados = total - df.dropDuplicates().count()
    print(f"\n=== CALIDAD: {etiqueta} ===")
    print("Filas:", total, "| Duplicados completos:", duplicados)
    print("Nulos / vacíos por columna:", nulos)
    return {"etapa": etiqueta, "filas": total, "duplicados": duplicados, "nulos": nulos}

calidad_antes = resumen_calidad(df_ventas_unificado, "Antes de limpiar")

In [ ]:
# Eliminamos duplicados completos ANTES de transformar, para no confundirlos
# con registros que solo se parecen después de normalizar el texto.
df_sin_duplicados = df_ventas_unificado.dropDuplicates()

categoria_normalizada = F.trim(F.col("categoria"))
precio_texto = F.regexp_replace(F.trim(F.col("precio_unitario")), r"[^0-9,.-]", "")
# En estos tres CSV los precios son enteros (pesos chilenos) con o sin '$'.
# La sustitución de coma decimal se deja explícita por robustez.
precio_normalizado = F.regexp_replace(precio_texto, ",", ".")

# Los CSV combinan yyyy-MM-dd, yyyy/MM/dd, dd/MM/yyyy y dd-MM-yyyy.
# try_to_date no está disponible en todas las versiones; usamos to_date
# después de identificar el patrón, evitando errores de ANSI por formatos mixtos.
fecha_original = F.trim(F.col("fecha"))
fecha_iso = (
    F.when(fecha_original.rlike(r"^\d{4}[-/]\d{2}[-/]\d{2}$"),
           F.regexp_replace(fecha_original, "/", "-"))
     .when(fecha_original.rlike(r"^\d{2}[-/]\d{2}[-/]\d{4}$"),
           F.concat(F.substring(fecha_original, 7, 4), F.lit("-"),
                    F.substring(fecha_original, 4, 2), F.lit("-"),
                    F.substring(fecha_original, 1, 2)))
     .otherwise(F.lit(None).cast("string"))
)

# Nombres de producto: minúsculas, un solo espacio y sin tildes.
nombre_base = F.translate(
    F.lower(F.regexp_replace(F.trim(F.col("producto_nombre")), r"\s+", " ")),
    "áéíóúü", "aeiouu")
# Sinónimos detectados en el diagnóstico: distintas formas de escribir el mismo artículo.
SINONIMOS_PRODUCTO = [
    (r"^laptop gam(er|ing)$", "laptop gamer"),
    (r"^(memoria )?ram 16 ?gb$", "memoria ram 16gb"),
    (r"^(disco )?ssd 1 ?tb$", "ssd 1tb"),
    (r"^monitor 27", "monitor 27 pulgadas"),
    (r"^router wi-?fi$", "router wifi"),
]
nombre_producto = nombre_base
for patron, canonico in SINONIMOS_PRODUCTO:
    nombre_producto = F.when(nombre_base.rlike(patron), F.lit(canonico)).otherwise(nombre_producto)

df_limpieza = (df_sin_duplicados
    .withColumn("categoria", F.when(F.col("categoria").isNull() | (categoria_normalizada == ""),
                                     F.lit("Sin categoría")).otherwise(categoria_normalizada))
    .withColumn("producto_nombre", nombre_producto)
    .withColumn("fecha_estandarizada", fecha_iso)
    .withColumn("fecha", F.expr("try_cast(fecha_estandarizada AS DATE)"))
    .drop("fecha_estandarizada")
    .withColumn("precio_normalizado_tmp", precio_normalizado)
    .withColumn("precio_unitario", F.expr("try_cast(precio_normalizado_tmp AS DOUBLE)"))
    .drop("precio_normalizado_tmp")
)
df_limpieza = df_limpieza.withColumn("cantidad", F.expr("try_cast(cantidad AS INT)"))

print("Nombres de producto distintos | originales:",
      df_sin_duplicados.select("producto_nombre").distinct().count(),
      "| solo minúsculas:",
      df_sin_duplicados.select(F.lower(F.trim("producto_nombre"))).distinct().count(),
      "| normalizados:", df_limpieza.select("producto_nombre").distinct().count())
df_limpieza.groupBy("producto_nombre").count().orderBy("producto_nombre").show(truncate=False)

print("Después de estandarizar:")
df_limpieza.select("fecha", "producto_nombre", "categoria", "cantidad", "precio_unitario").show(10, truncate=False)

**Control de interpretación.** Las fechas se transforman en formato ISO (`yyyy-MM-dd`). El precio se limpia antes de convertirlo; si aparecieran valores no numéricos en otro archivo, debemos investigarlos antes de incorporarlos al análisis.

In [ ]:
# Comprobamos que los campos esenciales sean utilizables.
invalidos_formato = df_limpieza.filter(
    F.col("fecha").isNull() | F.col("precio_unitario").isNull() |
    F.col("cantidad").isNull()
)
print("Registros con fecha, precio o cantidad no interpretables:", invalidos_formato.count())
invalidos_formato.show(10, truncate=False)

## 3. Transformación y validación (ID4.1, ID4.3)
Excluimos cantidades menores o iguales a cero y registros sin fecha o precio válido. Calculamos `precio_total` (`cantidad * precio_unitario`) y `dia_semana` (en español, a partir de la fecha estandarizada), y verificamos que no existan totales negativos.

**Valores atípicos de precio.** Aplicamos dos reglas y **separamos** los registros que no las cumplen en un conjunto de cuarentena (`df_cuarentena`), en vez de borrarlos:
1. **IQR:** precio fuera de `[Q1 − 1,5·IQR, Q3 + 1,5·IQR]`. Detecta los valores centinela `9.999.990` y `8.888.888`, que aparecen en productos tan distintos como un mouse y una laptop: no son precios reales, sino marcadores de error de carga.
2. **Precio mínimo de negocio:** precio menor a $1.000. El límite inferior del IQR es negativo, por lo que nunca detectaría precios de `$1` o `$50` (por ejemplo, una laptop a $1). El precio legítimo más bajo del catálogo es $25.990.

Las consultas SQL se calculan solo sobre los registros válidos; la cuarentena queda disponible para auditoría.

In [ ]:
df_ventas_filtrado = df_limpieza.filter(
    (F.col("cantidad") > 0) & (F.col("precio_unitario") >= 0) &
    F.col("fecha").isNotNull() & F.col("precio_unitario").isNotNull()
)
filas_excluidas = df_limpieza.count() - df_ventas_filtrado.count()
print("Filas excluidas por reglas de negocio (cantidad <= 0 o datos no interpretables):", filas_excluidas)

# dayofweek: 1 = domingo ... 7 = sábado
DIAS_SEMANA = ["domingo", "lunes", "martes", "miércoles", "jueves", "viernes", "sábado"]
df_ventas_transformado = (df_ventas_filtrado
    .withColumn("precio_total", F.round(F.col("cantidad") * F.col("precio_unitario"), 2))
    .withColumn("dia_semana", F.element_at(F.array(*[F.lit(d) for d in DIAS_SEMANA]),
                                           F.dayofweek(F.col("fecha")))))

negativos = df_ventas_transformado.filter(F.col("precio_total") < 0).count()
assert negativos == 0, "Se detectaron precios totales negativos"
print("Totales negativos:", negativos, "| Validación: OK")
print("Sucursales distintas:", df_ventas_transformado.select("sucursal").distinct().count())
df_ventas_transformado.select("sucursal").distinct().show(truncate=False)
print("Categorías distintas:", df_ventas_transformado.select("categoria").distinct().count())
df_ventas_transformado.select("categoria").distinct().orderBy("categoria").show(truncate=False)

In [ ]:
# Regla 1 - IQR: atípico si precio < Q1 - 1.5*IQR o > Q3 + 1.5*IQR.
q1, q3 = df_ventas_transformado.approxQuantile("precio_unitario", [0.25, 0.75], 0.0)
iqr = q3 - q1
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr
# Regla 2 - mínimo de negocio: el límite inferior del IQR es negativo y no detecta precios de $1 o $50.
PRECIO_MINIMO_VALIDO = 1000  # CLP; el precio legítimo más bajo del catálogo es $25.990

df_ventas_final = (df_ventas_transformado
    .withColumn("es_outlier_precio",
                (F.col("precio_unitario") < F.lit(limite_inferior)) |
                (F.col("precio_unitario") > F.lit(limite_superior)))
    .withColumn("es_precio_irrisorio", F.col("precio_unitario") < F.lit(PRECIO_MINIMO_VALIDO))
    .withColumn("motivo_cuarentena",
                F.when(F.col("es_outlier_precio"), F.lit("precio sobre límite IQR"))
                 .when(F.col("es_precio_irrisorio"), F.lit("precio bajo mínimo"))))

df_ventas_validas = df_ventas_final.filter(F.col("motivo_cuarentena").isNull())
df_cuarentena = df_ventas_final.filter(F.col("motivo_cuarentena").isNotNull())

print(f"Q1={q1:,.2f} | Q3={q3:,.2f} | IQR={iqr:,.2f}")
print(f"Límites IQR: [{limite_inferior:,.2f}, {limite_superior:,.2f}] | Precio mínimo de negocio: {PRECIO_MINIMO_VALIDO:,}")
print("Registros válidos para análisis:", df_ventas_validas.count(), "| En cuarentena:", df_cuarentena.count())
df_cuarentena.groupBy("motivo_cuarentena").count().show(truncate=False)
df_cuarentena.select(
    "id_transaccion", "sucursal", "producto_nombre", "cantidad", "precio_unitario",
    F.col("precio_total").cast("bigint").alias("precio_total"), "motivo_cuarentena"
).orderBy(F.desc("precio_unitario")).show(20, truncate=False)

monto_total = df_ventas_final.agg(F.sum("precio_total")).first()[0]
monto_cuarentena = df_cuarentena.agg(F.sum("precio_total")).first()[0]
print(f"Monto en cuarentena: {monto_cuarentena:,.0f} de {monto_total:,.0f} "
      f"({monto_cuarentena / monto_total:.1%} del total)")

**Hallazgo: los 14 registros en cuarentena concentran cerca del 60 % del monto total.** Solo 7 ventas con precios centinela suman más que las 260 ventas válidas juntas. Si se mantuvieran en las consultas, el ranking de ventas mostraría transacciones ficticias y alteraría la comparación entre sucursales.

**Coherencia producto–categoría.** La celda siguiente muestra que cada producto aparece en varias categorías distintas (por ejemplo, una laptop clasificada como *Audio*). Esto indica que la columna `categoria` no es confiable en el origen. No la reclasificamos, porque no hay una tabla maestra de productos que respalde la corrección, pero lo consideramos al interpretar la Consulta 3. Por la misma razón, `id_producto` no identifica productos de forma única.

In [ ]:
# Diagnóstico de consistencia: ¿un producto tiene siempre la misma categoría?
(df_ventas_validas.groupBy("producto_nombre")
    .agg(F.countDistinct("categoria").alias("categorias_distintas"),
         F.countDistinct("id_producto").alias("ids_producto_distintos"),
         F.count("*").alias("ventas"))
    .orderBy("producto_nombre")
    .show(truncate=False))

### 3.1 Informe comparativo de calidad
Contrastamos el dataset bruto con el dataset final válido: filas, duplicados completos y valores nulos/vacíos por columna. También mostramos cuántas filas se pierden en cada etapa, para que cada exclusión quede trazada.

In [ ]:
# motivo_cuarentena es NULL por definición en los registros válidos; se excluye del conteo de nulos.
calidad_despues = resumen_calidad(df_ventas_validas.drop("motivo_cuarentena"), "Después de limpiar y validar")
columnas_origen = df_ventas_unificado.columns
comparacion = [(c, calidad_antes["nulos"].get(c, 0), calidad_despues["nulos"].get(c, 0)) for c in columnas_origen]
df_comparacion_nulos = spark.createDataFrame(comparacion, ["columna", "nulos_antes", "nulos_despues"])
print("Comparación de nulos por columna:")
df_comparacion_nulos.show(truncate=False)
print("Resumen de filas y duplicados:")
spark.createDataFrame([
    ("Antes", calidad_antes["filas"], calidad_antes["duplicados"]),
    ("Después", calidad_despues["filas"], calidad_despues["duplicados"])
], ["etapa", "filas", "duplicados_completos"]).show(truncate=False)

print("Trazabilidad de filas por etapa:")
spark.createDataFrame([
    ("1. Unificado (bruto)", filas_unificadas),
    ("2. Sin duplicados completos", df_sin_duplicados.count()),
    ("3. Con cantidad > 0, fecha y precio válidos", df_ventas_transformado.count()),
    ("4. Válido para análisis (sin cuarentena)", df_ventas_validas.count()),
], ["etapa", "filas"]).show(truncate=False)
print("Filas eliminadas en total:", calidad_antes["filas"] - calidad_despues["filas"])
assert calidad_despues["duplicados"] == 0, "Persisten duplicados completos"

## 4. Consultas con Spark SQL (ID3.3)
Creamos la vista temporal `ventas_limpias` con los **registros válidos** (sin cuarentena) y una vista `ventas_auditoria` con todos los registros y sus marcas, para medir el efecto de la depuración. Las tres consultas solicitadas se ejecutan sobre `ventas_limpias`.

In [ ]:
df_ventas_validas.createOrReplaceTempView("ventas_limpias")
df_ventas_final.createOrReplaceTempView("ventas_auditoria")
print("Vistas SQL creadas: ventas_limpias (", df_ventas_validas.count(),
      "filas ) y ventas_auditoria (", df_ventas_final.count(), "filas )")

### Consulta 1: las 10 ventas con mayor importe
**Interpretación:** las ventas de mayor valor corresponden a 10 unidades de teclado mecánico a $449.990 (**$4.499.900**, en Viña del Mar y Santiago), seguidas de laptops gamer y webcams. Ninguna supera los $4,5 millones. Antes de depurar, los siete primeros lugares los ocupaban precios centinela de hasta $79,9 millones.

In [ ]:
consulta_1 = spark.sql("""
SELECT id_transaccion, fecha, sucursal, producto_nombre,
       CAST(precio_total AS BIGINT) AS precio_total
FROM ventas_limpias
ORDER BY precio_total DESC, id_transaccion
LIMIT 10
""")
consulta_1.show(10, truncate=False)

### Consulta 2: monto total y cantidad de transacciones por sucursal
**Interpretación:** **Santiago** lidera con unos $108,3 millones en 105 transacciones, seguido de **Viña del Mar** (≈ $89,0 millones; 89) y **Concepción** (≈ $56,1 millones; 66). El ticket promedio es similar en Santiago y Viña del Mar (≈ $1,0 millón) y menor en Concepción (≈ $850 mil). Sin la depuración, Concepción aparecía segunda solo por una venta ficticia de $79,9 millones.

In [ ]:
consulta_2 = spark.sql("""
SELECT sucursal,
       CAST(ROUND(SUM(precio_total), 0) AS BIGINT) AS ventas_totales,
       COUNT(*) AS cantidad_transacciones,
       CAST(ROUND(AVG(precio_total), 0) AS BIGINT) AS ticket_promedio
FROM ventas_limpias
GROUP BY sucursal
ORDER BY ventas_totales DESC
""")
consulta_2.show(truncate=False)

### Consulta 3: precio promedio, mínimo y máximo por categoría
**Interpretación:** todas las categorías tienen el mismo rango de precios ($25.990 a $449.990) y promedios parecidos (≈ $147 mil a $201 mil). Esto es coherente con el hallazgo de la sección 3: la `categoria` está asignada de forma inconsistente en el origen, por lo que **no permite distinguir segmentos de precio**. `Sin categoría` representa valores imputados, no una clasificación comercial confirmada.

In [ ]:
consulta_3 = spark.sql("""
SELECT categoria,
       ROUND(AVG(precio_unitario), 2) AS precio_promedio,
       CAST(MIN(precio_unitario) AS BIGINT) AS precio_minimo,
       CAST(MAX(precio_unitario) AS BIGINT) AS precio_maximo,
       COUNT(*) AS transacciones
FROM ventas_limpias
GROUP BY categoria
ORDER BY categoria
""")
consulta_3.show(truncate=False)

### Consulta complementaria: efecto de la depuración por sucursal
Comparamos las ventas con y sin los registros en cuarentena, usando la vista `ventas_auditoria`. Muestra por qué no basta con **marcar** los atípicos: si se suman, cambian las conclusiones del negocio.

In [ ]:
consulta_impacto = spark.sql("""
SELECT sucursal,
       CAST(ROUND(SUM(precio_total), 0) AS BIGINT) AS ventas_con_anomalos,
       CAST(ROUND(SUM(CASE WHEN motivo_cuarentena IS NULL THEN precio_total ELSE 0 END), 0) AS BIGINT) AS ventas_validas,
       SUM(CASE WHEN motivo_cuarentena IS NOT NULL THEN 1 ELSE 0 END) AS registros_en_cuarentena
FROM ventas_auditoria
GROUP BY sucursal
ORDER BY ventas_validas DESC
""")
consulta_impacto.show(truncate=False)

## 5. Conclusiones
- **Integración:** la unión por nombre de columna conserva las 315 filas originales (105 + 125 + 85), verificado con un conteo antes y después.
- **Limpieza:** eliminamos 15 duplicados completos, imputamos 46 categorías vacías como `Sin categoría`, estandarizamos cuatro formatos de fecha y precios con `$`, y reducimos 17 variantes de nombre a 10 productos reales.
- **Validación:** excluimos 26 registros con cantidad ≤ 0 y separamos en cuarentena 14 precios no plausibles: 7 centinelas (`9.999.990` y `8.888.888`) detectados con IQR y 7 precios bajo $1.000 que el IQR no podía detectar. El dataset final para análisis tiene **260 ventas válidas**.
- **Hallazgo principal:** los 14 registros en cuarentena representaban cerca del **60 % del monto total**. Marcar los atípicos no es suficiente si luego se suman en las consultas: hacerlo invertía el orden entre Concepción y Viña del Mar y llenaba el ranking de ventas ficticias.
- **Resultados:** Santiago lidera en ventas (≈ $108,3 millones), seguido de Viña del Mar (≈ $89,0 millones) y Concepción (≈ $56,1 millones).
- **Limitación:** `categoria` e `id_producto` son inconsistentes en el origen (un mismo producto aparece en varias categorías), por lo que la Consulta 3 no permite comparar segmentos de precio. Recomendamos corregirlos con una tabla maestra de productos antes de usarlos en decisiones.

**Entrega:** ejecutar todas las celdas en Databricks, revisar los conteos, exportar el notebook con el nombre `mcdi502_s4_grupo2` y comprimirlo en un `.zip` para Canvas.

In [ ]:
print("=== CONTROL FINAL ===")
print("Archivos de origen:", len(archivos))
print("Filas iniciales:", filas_unificadas)
print("Filas tras reglas de negocio:", df_ventas_final.count())
print("Filas válidas para análisis:", df_ventas_validas.count(), "| En cuarentena:", df_cuarentena.count())
print("Productos distintos:", df_ventas_validas.select("producto_nombre").distinct().count())
print("Duplicados finales:", df_ventas_validas.count() - df_ventas_validas.dropDuplicates().count())
print("Precios totales negativos:", df_ventas_validas.filter(F.col("precio_total") < 0).count())
precio_min, precio_max = df_ventas_validas.agg(F.min("precio_unitario"), F.max("precio_unitario")).first()
print(f"Rango de precios válidos: {precio_min:,.0f} – {precio_max:,.0f}")
assert df_ventas_validas.select("sucursal").distinct().count() == 3
assert precio_min >= PRECIO_MINIMO_VALIDO and precio_max <= limite_superior, "Hay precios no plausibles en ventas_limpias"
print("Consultas SQL: 3 de 3 (+ 1 complementaria)")
print("Validación final completada")

## Declaración de uso de inteligencia artificial
De acuerdo con los lineamientos de la UNAB para el uso de IA generativa, declaramos que utilizamos un asistente de IA (Claude, de Anthropic) para revisar la calidad de los datos, detectar inconsistencias y proponer mejoras al código y a la documentación de este notebook. El grupo revisó, ejecutó y validó todas las transformaciones y resultados, y es responsable de su contenido.